# Training Demo

This notebook demonstrates training the **StrokeGAT** model for stroke lesion
detection on brain MRI graphs.

## Architecture

- **3 GAT layers** with residual connections
- **8 attention heads** per layer (multi-head attention, Eq. 2-3)
- **Hidden dimension:** 128 per head
- **Output:** 3-class node classification (normal / penumbra / core)

## Training Setup

- **Loss:** Composite loss = weighted cross-entropy + Dice loss (Eq. 10-11)
- **Optimizer:** AdamW with cosine annealing LR schedule
- **Metrics:** Composite score = 0.4 * Dice + 0.3 * AUC + 0.3 * Sensitivity
- **5-fold cross-validation** with stratified splits

> **Note:** This demo uses a reduced epoch count for quick iteration.
> Full training typically uses 200+ epochs.

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import torch
import numpy as np
import matplotlib.pyplot as plt

from stroke_gat.config import Config
from stroke_gat.data.loader import StrokeDataModule
from stroke_gat.models.gat import StrokeGAT
from stroke_gat.training.trainer import Trainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load config with reduced epochs for demo
cfg = Config.from_yaml("../configs/default.yaml")

# Override for quick demo run
cfg.training.epochs = 20
cfg.training.patience = 10  # early stopping patience

print("Training configuration:")
print(f"  Epochs:          {cfg.training.epochs}")
print(f"  Batch size:      {cfg.training.batch_size}")
print(f"  Learning rate:   {cfg.training.lr}")
print(f"  Weight decay:    {cfg.training.weight_decay}")
print(f"  Patience:        {cfg.training.patience}")
print(f"  Device:          {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Setup data module with train/val/test splits
data_module = StrokeDataModule(cfg)
data_module.setup()

print(f"Dataset splits:")
print(f"  Train: {len(data_module.train_dataset):>5} graphs")
print(f"  Val:   {len(data_module.val_dataset):>5} graphs")
print(f"  Test:  {len(data_module.test_dataset):>5} graphs")

# Peek at a single training sample
sample = data_module.train_dataset[0]
print(f"\nSample graph:")
print(f"  Nodes: {sample.num_nodes}, Edges: {sample.num_edges}")
print(f"  Feature dim: {sample.x.shape[1]}")
print(f"  Labels: {torch.bincount(sample.y)}")

In [ ]:
# Create the StrokeGAT model
model = StrokeGAT(
    in_channels=cfg.model.in_channels,
    hidden_channels=cfg.model.hidden_channels,
    out_channels=cfg.model.num_classes,
    num_layers=cfg.model.num_layers,
    num_heads=cfg.model.num_heads,
    dropout=cfg.model.dropout,
)

# Print architecture summary
print("StrokeGAT Architecture")
print("=" * 50)
print(model)
print("=" * 50)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:>10,}")
print(f"Trainable parameters: {trainable_params:>10,}")

In [ ]:
# Create trainer and run training
trainer = Trainer(
    cfg=cfg,
    model=model,
    data_module=data_module,
)

print("Starting training...\n")
history = trainer.fit()

print(f"\nTraining complete!")
print(f"Best validation composite score: {history['best_val_score']:.4f}")
print(f"Best epoch: {history['best_epoch']}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs = range(1, len(history["train_loss"]) + 1)

# Loss
axes[0, 0].plot(epochs, history["train_loss"], "b-", label="Train", linewidth=2)
axes[0, 0].plot(epochs, history["val_loss"], "r-", label="Val", linewidth=2)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Composite Score
axes[0, 1].plot(epochs, history["val_composite"], "g-", linewidth=2)
axes[0, 1].axhline(history["best_val_score"], color="red", linestyle="--",
                    alpha=0.7, label=f"Best: {history['best_val_score']:.4f}")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Composite Score")
axes[0, 1].set_title("Validation Composite Score")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Dice Score
axes[1, 0].plot(epochs, history["val_dice"], "m-", linewidth=2)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Dice Score")
axes[1, 0].set_title("Validation Dice Score")
axes[1, 0].grid(True, alpha=0.3)

# AUC
axes[1, 1].plot(epochs, history["val_auc"], "c-", linewidth=2)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("AUC")
axes[1, 1].set_title("Validation AUC")
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle("Training Curves", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Run test evaluation with best checkpoint
print("Evaluating on test set with best checkpoint...\n")
test_results = trainer.test()

# Print results as a formatted table
print(f"{'Metric':<25} {'Value':>10}")
print("=" * 37)
for metric, value in sorted(test_results.items()):
    if isinstance(value, float):
        print(f"{metric:<25} {value:>10.4f}")
    else:
        print(f"{metric:<25} {str(value):>10}")

print(f"\n{'=' * 37}")
print(f"{'Composite Score':<25} {test_results.get('composite_score', 0.0):>10.4f}")

## Discussion

**Expected Performance (with full training, 200 epochs):**

| Metric           | Expected Range |
|------------------|---------------|
| Dice (Penumbra)  | 0.55 - 0.65   |
| Dice (Core)      | 0.60 - 0.72   |
| AUC              | 0.82 - 0.90   |
| Sensitivity      | 0.70 - 0.82   |
| Composite Score  | 0.68 - 0.76   |

**Hyperparameter Tuning Tips:**

1. **Learning rate:** Start with 1e-3 and reduce; cosine annealing handles decay
2. **Number of heads:** 8 heads work well; more heads provide diminishing returns
3. **Hidden channels:** 128 balances capacity and speed; 256 may help on larger graphs
4. **Dropout:** 0.1-0.3 range; higher values needed if overfitting is observed
5. **Class weights:** Adjust based on the degree of class imbalance in training set

Next: See `05_attention_attribution_gallery.ipynb` for model interpretability.